## Confusion Matrix Galaxy Images
For each category (TN/FP/FN/TP) grab galaxy FITS files and make 2×2 grids.

In [ ]:
import pandas as pd
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from pathlib import Path

CSV_FILE          = "rf_predictions.csv"
FITS_FOLDER       = "test"
OUTPUT_BASE       = "confusion_matrix_galaxies"
GALAXIES_PER_IMG  = 4


In [ ]:
def galaxy_name_to_fits(name, folder):
    stem = name.replace(' ', '_')
    for pat in [f"norm_resized_{stem}.fits",
                f"norm_resized_{stem.replace('_','-')}.fits",
                f"norm_resized_{stem.replace('-','_')}.fits"]:
        p = Path(folder) / pat
        if p.exists(): return str(p)
    return None

def load_fits(path):
    try:
        with fits.open(path) as hdul:
            d = hdul[0].data
            return d[0] if d.ndim > 2 and d.shape[0] < d.shape[-1] else np.squeeze(d)
    except: return None

def make_grid(batch, out_path, title):
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    axes = axes.flatten()
    for i, (name, data, pred, true, conf) in enumerate(batch[:4]):
        ax = axes[i]
        if data is not None:
            lo, hi = np.percentile(data, [1, 99])
            im = ax.imshow(data, cmap='gray', origin='lower', vmin=lo, vmax=hi)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            p_str = "Barred" if pred == 1 else "Unbarred"
            t_str = "Barred" if true == 1 else "Unbarred"
            ax.set_title(f"{name}\nTrue: {t_str}, Pred: {p_str}\nConf: {conf:.3f}", fontsize=10)
        else:
            ax.text(0.5, 0.5, f"{name}\n(not found)", ha='center', va='center')
            ax.set_title(f"{name} (missing)", fontsize=10)
        ax.axis('off')
    for j in range(len(batch), 4):
        axes[j].axis('off')
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  saved: {out_path}")


In [ ]:
df = pd.read_csv(CSV_FILE)
print(f"Loaded {len(df)} predictions")

# standardise column names across different CSV formats
if 'RF_Predicted_Label' in df.columns:
    df['Predicted_Label'] = df['RF_Predicted_Label']
    df['Confidence']      = df.get('RF_Confidence', df.get('RF_Probability_Class_1', 0.5))
elif 'Confidence' not in df.columns:
    df['Confidence'] = df.get('Probability_Barred', 0.5)

categories = {
    'TN': ('True_Negative_Unbarred',  0, 0),
    'FP': ('False_Positive_Unbarred', 0, 1),
    'FN': ('False_Negative_Barred',   1, 0),
    'TP': ('True_Positive_Barred',    1, 1),
}

base = Path(OUTPUT_BASE); base.mkdir(exist_ok=True)

for key, (folder_name, true_val, pred_val) in categories.items():
    cat_dir = base / folder_name; cat_dir.mkdir(exist_ok=True)
    subset  = df[(df['True_Label'] == true_val) & (df['Predicted_Label'] == pred_val)]
    print(f"\n{folder_name}: {len(subset)} galaxies")
    if len(subset) == 0: continue

    batch = []; n = 0
    for _, row in subset.iterrows():
        gname    = row['Target_Name']
        fpath    = galaxy_name_to_fits(gname, FITS_FOLDER)
        data     = load_fits(fpath) if fpath else None
        if fpath:
            print(f"  found: {gname}")
        else:
            print(f"  missing: {gname}")
        batch.append((gname, data, int(row['Predicted_Label']),
                      int(row['True_Label']), float(row['Confidence'])))
        if len(batch) == GALAXIES_PER_IMG:
            n += 1
            make_grid(batch, cat_dir / f"{key}_batch_{n:03d}.png",
                      folder_name.replace('_',' '))
            batch = []
    if batch:
        n += 1
        make_grid(batch, cat_dir / f"{key}_batch_{n:03d}.png",
                  folder_name.replace('_',' '))

print("\nAll done!")
